# Gradient Descent: Complete Guide

## Notebook 1: Gradient Descent from Scratch (Complete)

```markdown
# Gradient Descent: From Scratch to Mastery

## 1. Introduction

### Why Optimization Matters
Optimization is the backbone of machine learning. Every ML algorithm - from simple linear regression to deep neural networks - relies on optimization to find the best parameters that minimize error. Gradient Descent is the most fundamental optimization algorithm.

```mermaid
graph TD
    A[Machine Learning Problem] --> B[Define Model]
    B --> C[Define Loss Function]
    C --> D[Choose Optimizer]
    D --> E[Gradient Descent]
    D --> F[Other Optimizers]
    
    E --> G[Find Optimal Parameters]
    F --> G
    
    style E fill:#c8e6c9
    style G fill:#e3f2fd
```

### Loss Functions
A loss function measures how wrong our model's predictions are. The goal is to minimize this error.

**Common Loss Functions:**

```python
import numpy as np
import matplotlib.pyplot as plt

# Mean Squared Error (MSE) - Most common for regression
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# Mean Absolute Error (MAE) - Robust to outliers
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Huber Loss - Combines MSE and MAE
def huber_loss(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    is_small_error = np.abs(error) <= delta
    squared_loss = 0.5 * error**2
    linear_loss = delta * (np.abs(error) - 0.5 * delta)
    return np.mean(np.where(is_small_error, squared_loss, linear_loss))

# Visualize different loss functions
errors = np.linspace(-3, 3, 100)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(errors, [mse(0, e) for e in errors], label='MSE')
ax.plot(errors, [mae(0, e) for e in errors], label='MAE')
ax.plot(errors, [huber_loss(0, e) for e in errors], label='Huber')
ax.set_xlabel('Error')
ax.set_ylabel('Loss')
ax.set_title('Comparison of Loss Functions')
ax.legend()
ax.grid(True)
plt.show()
```

### Convex vs Non-convex Functions

```mermaid
graph LR
    A[Loss Functions] --> B[Convex]
    A --> C[Non-convex]
    
    B --> D[One Global Minimum]
    B --> E[Easy to Optimize]
    B --> F[Guaranteed Convergence]
    
    C --> G[Multiple Local Minima]
    C --> H[Harder to Optimize]
    C --> I[May Get Stuck]
    
    style B fill:#c8e6c9
    style C fill:#ffebee
```

## 2. Summary of Gradient Descent

Gradient Descent is an iterative optimization algorithm used to minimize a function by moving in the direction of steepest descent.

### Key Concepts:
- **Loss Function**: Measures how well the model performs
- **Gradient**: Vector of partial derivatives pointing uphill
- **Learning Rate**: Controls step size
- **Convergence**: When algorithm reaches minimum

```mermaid
graph TD
    A[Start] --> B[Initialize Parameters]
    B --> C[Compute Gradient]
    C --> D[Update Parameters: θ = θ - α∇J]
    D --> E[Check Convergence]
    E -->|No| C
    E -->|Yes| F[Optimal Parameters]
    
    style A fill:#e3f2fd
    style F fill:#c8e6c9
```

## 3. What is Gradient Descent?

Gradient descent is like being blindfolded on a mountain and needing to find the lowest point. By feeling the slope (gradient) and stepping in the steepest downward direction, you eventually reach the valley.

### Mathematical Representation:
```
θ_new = θ_old - α · ∇J(θ_old)
```
Where:
- α = Learning rate
- ∇J(θ) = Gradient of loss function

## 4. Intuition for Gradient Descent

### a. Intuition: b_new = b_old - slope

Imagine you're standing on a hillside:
- If slope is positive, you need to move LEFT (subtract)
- If slope is negative, you need to move RIGHT (add)
- The steeper the slope, the larger the step

### b. When to Stop

1. **Maximum Iterations**: Pre-set limit (epochs)
2. **Tolerance**: Stop when loss change is minimal
3. **Gradient Norm**: Stop when gradient ≈ 0

```mermaid
graph TD
    A[Convergence Criteria] --> B[Max Epochs]
    A --> C[Tolerance Met]
    A --> D[Gradient Zero]
    
    B --> E[Stop Training]
    C --> E
    D --> E
```

## 5. Mathematical Formulation

### Simple Linear Regression:
Given points (x₁,y₁), (x₂,y₂), ..., (xₙ,yₙ), find y = mx + b that minimizes:

```
Loss = (1/n) Σ(yᵢ - (mxᵢ + b))²
```

### Gradients:
```
∂Loss/∂m = -(2/n) Σ(yᵢ - mxᵢ - b) · xᵢ
∂Loss/∂b = -(2/n) Σ(yᵢ - mxᵢ - b)
```

### Update Rules:
```
m_new = m_old - α · ∂Loss/∂m
b_new = b_old - α · ∂Loss/∂b
```

### Convergence Proof Intuition:
The algorithm converges because:
1. Each step reduces loss (if α is small enough)
2. Loss function is bounded below by 0
3. Therefore, sequence converges (Monotone Convergence Theorem)

## 6. Code Demo: Setup and Data

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

# Generate synthetic dataset
np.random.seed(42)
X, y = make_regression(
    n_samples=100,
    n_features=1,
    n_informative=1,
    noise=20,
    random_state=13
)

# Fit OLS as benchmark
reg = LinearRegression()
reg.fit(X, y)

print(f"OLS Slope (m): {reg.coef_[0]:.4f}")
print(f"OLS Intercept (b): {reg.intercept_:.4f}")

# Visualize data
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, alpha=0.6, label='Data points')
ax.plot(X, reg.predict(X), color='red', linewidth=2, label='OLS Line')
ax.set_xlabel('X')
ax.set_ylabel('y')
ax.set_title('Regression Data with OLS Line')
ax.legend()
ax.grid(True)
plt.show()
```

## 7. Building Custom Gradient Descent Class

```python
class GDRegressor:
    """
    Gradient Descent Regressor from Scratch
    
    Parameters:
    -----------
    learning_rate : float, default=0.001
        Step size for each iteration
    epochs : int, default=50
        Number of iterations
    """
    
    def __init__(self, learning_rate=0.001, epochs=50):
        self.m = 100  # Initial guess for slope
        self.b = -120  # Initial guess for intercept
        self.lr = learning_rate
        self.epochs = epochs
        self.history = {'m': [], 'b': [], 'loss': []}
    
    def fit(self, X, y):
        """
        Train the model using gradient descent
        
        Parameters:
        -----------
        X : array-like
            Training features
        y : array-like
            Target values
        """
        n = len(X)
        X_flat = X.ravel()
        
        for epoch in range(self.epochs):
            # Calculate predictions
            y_pred = self.m * X_flat + self.b
            
            # Calculate gradients
            loss_slope_b = -2 * np.sum(y - y_pred)
            loss_slope_m = -2 * np.sum((y - y_pred) * X_flat)
            
            # Update parameters
            self.b = self.b - self.lr * loss_slope_b
            self.m = self.m - self.lr * loss_slope_m
            
            # Store history
            loss = np.mean((y - y_pred)**2)
            self.history['m'].append(self.m)
            self.history['b'].append(self.b)
            self.history['loss'].append(loss)
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: m={self.m:.4f}, b={self.b:.4f}, loss={loss:.4f}")
        
        return self
    
    def predict(self, X):
        """Make predictions"""
        return self.m * X + self.b

# Train custom model
gd = GDRegressor(learning_rate=0.001, epochs=50)
gd.fit(X, y)

print(f"\nFinal Parameters:")
print(f"GD m: {gd.m:.4f}")
print(f"GD b: {gd.b:.4f}")
print(f"OLS m: {reg.coef_[0]:.4f}")
print(f"OLS b: {reg.intercept_:.4f}")
```

## 8. Complete Visualization

### Loss Surface Visualization

```python
# Create loss surface
m_range = np.linspace(gd.m - 50, gd.m + 50, 50)
b_range = np.linspace(gd.b - 50, gd.b + 50, 50)
M, B = np.meshgrid(m_range, b_range)

Z = np.zeros_like(M)
X_flat = X.ravel()
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        y_pred = M[i,j] * X_flat + B[i,j]
        Z[i,j] = np.mean((y - y_pred)**2)

# 3D Surface Plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot surface
surf = ax.plot_surface(M, B, Z, cmap='viridis', alpha=0.7, label='Loss Surface')

# Plot GD path
m_history = gd.history['m']
b_history = gd.history['b']
loss_history = gd.history['loss']

ax.plot(m_history, b_history, loss_history, 'r-', linewidth=2, label='GD Path')
ax.scatter(m_history[0], b_history[0], loss_history[0], color='green', s=100, label='Start')
ax.scatter(m_history[-1], b_history[-1], loss_history[-1], color='red', s=100, label='End')

ax.set_xlabel('m (Slope)')
ax.set_ylabel('b (Intercept)')
ax.set_zlabel('Loss')
ax.set_title('Loss Surface with Gradient Descent Path')
ax.legend()
plt.show()
```

### Loss History Plot

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Full loss history
axes[0].plot(gd.history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss History')
axes[0].grid(True)

# Plot 2: Zoomed view (last 20 epochs)
axes[1].plot(gd.history['loss'][-20:])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (MSE)')
axes[1].set_title('Loss History (Last 20 Epochs)')
axes[1].grid(True)

plt.tight_layout()
plt.show()
```

### Animated Trajectory

```python
from IPython.display import HTML

def animate_gd(X, y, history):
    """Create animation of gradient descent convergence"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left plot: Data and regression line
    ax1.scatter(X, y, alpha=0.5, label='Data')
    x_line = np.linspace(X.min() - 1, X.max() + 1, 100)
    line, = ax1.plot([], [], 'r-', linewidth=2)
    ax1.set_xlabel('X')
    ax1.set_ylabel('y')
    ax1.set_title('Regression Line Evolution')
    ax1.legend()
    ax1.grid(True)
    
    # Right plot: Loss history
    loss_line, = ax2.plot([], [], 'b-', linewidth=2)
    ax2.set_xlim(0, len(history['loss']))
    ax2.set_ylim(0, max(history['loss']) * 1.1)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Loss Convergence')
    ax2.grid(True)
    
    def update(frame):
        # Update regression line
        m = history['m'][frame]
        b = history['b'][frame]
        y_line = m * x_line + b
        line.set_data(x_line, y_line)
        
        # Update loss plot
        x_data = list(range(frame + 1))
        y_data = history['loss'][:frame + 1]
        loss_line.set_data(x_data, y_data)
        
        ax1.set_title(f'Epoch {frame+1}: m={m:.3f}, b={b:.3f}')
        return line, loss_line
    
    anim = animation.FuncAnimation(
        fig, update,
        frames=len(history['loss']),
        interval=200,
        blit=True
    )
    
    plt.close()
    return anim

# Display animation
anim = animate_gd(X, y, gd.history)
HTML(anim.to_html5_video())
```

## 9. Effect of Learning Rate

```python
def test_learning_rates(X, y):
    """Compare different learning rates"""
    
    learning_rates = [0.0001, 0.001, 0.01, 0.1]
    results = {}
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for i, lr in enumerate(learning_rates):
        ax = axes[i//2, i%2]
        
        model = GDRegressor(learning_rate=lr, epochs=100)
        model.fit(X, y)
        
        ax.plot(model.history['loss'])
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'Learning Rate = {lr}')
        ax.grid(True)
        
        results[lr] = {
            'final_loss': model.history['loss'][-1],
            'convergence_speed': len(model.history['loss'])
        }
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print("\nLearning Rate Comparison:")
    print("-" * 40)
    for lr, data in results.items():
        print(f"LR = {lr}: Final Loss = {data['final_loss']:.4f}")
    
    return results

test_learning_rates(X, y)
```

### Visualization of Learning Rate Effects

```mermaid
graph TD
    subgraph "Learning Rate Effects"
        A[Small LR] --> B[Slow Convergence]
        A --> C[May Get Stuck]
        
        D[Optimal LR] --> E[Fast Convergence]
        D --> F[Reaches Minimum]
        
        G[Large LR] --> H[Oscillations]
        G --> I[Divergence]
    end
```

## 10. Effect of Initialization

```python
def test_initializations(X, y):
    """Test different initial parameter values"""
    
    initializations = [
        (-100, -100),
        (0, 0),
        (100, 100),
        (-100, 100)
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for i, (m_init, b_init) in enumerate(initializations):
        ax = axes[i//2, i%2]
        
        # Create model with custom initialization
        model = GDRegressor(learning_rate=0.001, epochs=50)
        model.m = m_init
        model.b = b_init
        model.fit(X, y)
        
        # Plot GD path on loss surface
        m_range = np.linspace(m_init - 50, m_init + 50, 30)
        b_range = np.linspace(b_init - 50, b_init + 50, 30)
        M, B = np.meshgrid(m_range, b_range)
        Z = np.zeros_like(M)
        for j in range(M.shape[0]):
            for k in range(M.shape[1]):
                y_pred = M[j,k] * X.ravel() + B[j,k]
                Z[j,k] = np.mean((y - y_pred)**2)
        
        ax.contour(M, B, Z, levels=20, cmap='viridis')
        ax.plot(model.history['m'], model.history['b'], 'r-', linewidth=2)
        ax.scatter(model.history['m'][0], model.history['b'][0], color='green', s=50, label='Start')
        ax.scatter(model.history['m'][-1], model.history['b'][-1], color='red', s=50, label='End')
        ax.set_xlabel('m')
        ax.set_ylabel('b')
        ax.set_title(f'Init: m={m_init}, b={b_init}')
        ax.legend()
        ax.grid(True)
    
    plt.tight_layout()
    plt.show()
```

## 11. Performing Gradient Descent with Both Parameters

```python
def gradient_descent_full(X, y, learning_rate=0.001, epochs=100):
    """
    Complete gradient descent for both m and b
    
    Returns:
        m, b, history of parameters and loss
    """
    m = 0
    b = 0
    history = {'m': [], 'b': [], 'loss': []}
    n = len(X)
    X_flat = X.ravel()
    
    for epoch in range(epochs):
        y_pred = m * X_flat + b
        
        # Gradients
        dm = -(2/n) * np.sum((y - y_pred) * X_flat)
        db = -(2/n) * np.sum(y - y_pred)
        
        # Update
        m -= learning_rate * dm
        b -= learning_rate * db
        
        # Store
        history['m'].append(m)
        history['b'].append(b)
        history['loss'].append(np.mean((y - y_pred)**2))
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: m={m:.4f}, b={b:.4f}, loss={history['loss'][-1]:.4f}")
    
    return m, b, history

# Run complete GD
m_final, b_final, history = gradient_descent_full(X, y, learning_rate=0.001, epochs=100)
print(f"\nFinal: m={m_final:.4f}, b={b_final:.4f}")
```

## 12. 3D Visualization of GD Path

```python
# Create 3D animation of the GD path
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot loss surface
m_range = np.linspace(-50, 50, 40)
b_range = np.linspace(-50, 50, 40)
M, B = np.meshgrid(m_range, b_range)
Z = np.zeros_like(M)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        y_pred = M[i,j] * X.ravel() + B[i,j]
        Z[i,j] = np.mean((y - y_pred)**2)

ax.plot_surface(M, B, Z, alpha=0.5, cmap='viridis')

# Plot path
m_hist = history['m']
b_hist = history['b']
loss_hist = []
for m, b in zip(m_hist, b_hist):
    y_pred = m * X.ravel() + b
    loss_hist.append(np.mean((y - y_pred)**2))

# Plot path with color gradient
for i in range(len(m_hist)-1):
    ax.plot([m_hist[i], m_hist[i+1]],
            [b_hist[i], b_hist[i+1]],
            [loss_hist[i], loss_hist[i+1]],
            'r-', alpha=0.8)

ax.scatter(m_hist[0], b_hist[0], loss_hist[0], color='green', s=100, label='Start')
ax.scatter(m_hist[-1], b_hist[-1], loss_hist[-1], color='red', s=100, label='End')

ax.set_xlabel('m (Slope)')
ax.set_ylabel('b (Intercept)')
ax.set_zlabel('Loss')
ax.set_title('Gradient Descent Path on Loss Surface')
ax.legend()
plt.show()
```

## 13. Effect of Loss Function Choice

```python
class GDWithCustomLoss:
    """Gradient Descent with customizable loss function"""
    
    def __init__(self, loss_func='mse', learning_rate=0.001, epochs=100):
        self.loss_func = loss_func
        self.lr = learning_rate
        self.epochs = epochs
        self.history = {'m': [], 'b': [], 'loss': []}
        
    def fit(self, X, y):
        m = 0
        b = 0
        n = len(X)
        X_flat = X.ravel()
        
        for epoch in range(self.epochs):
            y_pred = m * X_flat + b
            
            # Different gradients for different loss functions
            if self.loss_func == 'mse':
                dm = -(2/n) * np.sum((y - y_pred) * X_flat)
                db = -(2/n) * np.sum(y - y_pred)
                loss = np.mean((y - y_pred)**2)
            elif self.loss_func == 'mae':
                error = y - y_pred
                dm = -(1/n) * np.sum(np.sign(error) * X_flat)
                db = -(1/n) * np.sum(np.sign(error))
                loss = np.mean(np.abs(error))
            
            m -= self.lr * dm
            b -= self.lr * db
            
            self.history['m'].append(m)
            self.history['b'].append(b)
            self.history['loss'].append(loss)
        
        return m, b, self.history

# Compare loss functions
loss_functions = ['mse', 'mae']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, loss_func in enumerate(loss_functions):
    model = GDWithCustomLoss(loss_func=loss_func, learning_rate=0.001, epochs=100)
    m, b, history = model.fit(X, y)
    
    axes[i].plot(history['loss'])
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].set_title(f'{loss_func.upper()} Loss')
    axes[i].grid(True)

plt.tight_layout()
plt.show()
```

## 14. Effect of Data Characteristics

```python
def experiment_data_characteristics():
    """Test how data characteristics affect gradient descent"""
    
    configs = [
        {'n_samples': 50, 'noise': 10},
        {'n_samples': 50, 'noise': 50},
        {'n_samples': 500, 'noise': 10},
        {'n_samples': 500, 'noise': 50}
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for i, config in enumerate(configs):
        ax = axes[i//2, i%2]
        
        # Generate data
        X, y = make_regression(
            n_samples=config['n_samples'],
            n_features=1,
            noise=config['noise'],
            random_state=42
        )
        
        # Run gradient descent
        model = GDRegressor(learning_rate=0.001, epochs=100)
        model.fit(X, y)
        
        # Plot results
        ax.scatter(X, y, alpha=0.5)
        X_line = np.linspace(X.min(), X.max(), 100)
        ax.plot(X_line, model.predict(X_line), 'r-', linewidth=2)
        ax.set_title(f'Size={config["n_samples"]}, Noise={config["noise"]}')
        ax.set_xlabel('X')
        ax.set_ylabel('y')
        ax.grid(True)
    
    plt.tight_layout()
    plt.show()

experiment_data_characteristics()
```

## 15. Common Mistakes and Debugging

```python
# Mistake 1: Not Scaling Features
def demonstrate_feature_scaling():
    """Show importance of feature scaling"""
    
    # Generate data with different scales
    np.random.seed(42)
    X_scale = np.random.randn(100, 2)
    X_scale[:, 1] = X_scale[:, 1] * 100  # Feature 2 has larger scale
    y = 2 * X_scale[:, 0] + 0.5 * X_scale[:, 1] + np.random.randn(100)
    
    # Without scaling
    model_no_scale = BatchGDRegressor(learning_rate=0.001, epochs=100)
    model_no_scale.fit(X_scale, y)
    
    # With scaling
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_scale)
    model_scaled = BatchGDRegressor(learning_rate=0.001, epochs=100)
    model_scaled.fit(X_scaled, y)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(model_no_scale.loss_history)
    axes[0].set_title('Without Feature Scaling')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    
    axes[1].plot(model_scaled.loss_history)
    axes[1].set_title('With Feature Scaling')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

demonstrate_feature_scaling()
```

## 16. Interview Questions

### Q1: What is the difference between Batch GD, SGD, and Mini-Batch GD?

```mermaid
graph TD
    A[Gradient Descent Variants] --> B[Batch GD]
    A --> C[Stochastic GD]
    A --> D[Mini-Batch GD]
    
    B --> E[Uses ALL data]
    C --> F[Uses 1 sample]
    D --> G[Uses subset]
    
    E --> H[Accurate, Slow]
    F --> I[Fast, Noisy]
    G --> J[Balanced]
```

### Q2: Why is learning rate important?

Learning rate controls step size. Too small = slow convergence, too large = might diverge.

### Q3: What problems can gradient descent have?

1. **Local Minima**: Can get stuck in non-convex functions
2. **Saddle Points**: Zero gradient but not minimum
3. **Vanishing Gradients**: Gradients become very small
4. **Exploding Gradients**: Gradients become very large

## 17. Advantages and Limitations

### Advantages:
- Simple to implement
- Works for any differentiable function
- Scalable to large datasets
- Can be improved with momentum

### Limitations:
- Requires differentiable loss function
- Can get stuck in local minima
- Sensitive to learning rate
- May converge slowly

## 18. Complexity Analysis

- **Time Complexity**: O(n * epochs) per iteration
- **Space Complexity**: O(n + features)
- **Memory**: O(n) for storing data

## 19. Summary Notes

1. Gradient Descent minimizes loss by following negative gradient
2. Learning rate is the most important hyperparameter
3. Feature scaling is essential for good performance
4. Different variants trade off speed vs. accuracy
5. Convergence depends on learning rate and function landscape

## 20. Practice Questions

1. What happens if learning rate is too high?
2. Why do we subtract the gradient?
3. How does feature scaling affect GD?
4. What's the difference between convex and non-convex functions?
5. When would you use momentum in GD?

## 21. Exercises

### Exercise 1: Implement Adaptive Learning Rate
```python
def adaptive_lr_GD(X, y, initial_lr=0.01, epochs=100):
    """Implement GD with learning rate decay"""
    pass  # Your implementation
```

### Exercise 2: Add Momentum
```python
def momentum_GD(X, y, learning_rate=0.01, momentum=0.9, epochs=100):
    """Implement GD with momentum"""
    pass  # Your implementation
```
```

---

## Notebook 2: Batch Gradient Descent

```markdown
# Batch Gradient Descent: Complete Implementation

## 1. Why Batch Gradient Descent?

Batch Gradient Descent (BGD) computes the gradient using the **entire training dataset** for each update.

```mermaid
graph TD
    A[Batch GD Process] --> B[Initialize Parameters]
    B --> C[Forward Pass: All Data]
    C --> D[Compute Cost: All Data]
    D --> E[Backward Pass: All Data]
    E --> F[Update Parameters]
    F --> G{Converged?}
    G -->|No| C
    G -->|Yes| H[Final Model]
    
    style C fill:#e3f2fd
    style D fill:#e3f2fd
    style E fill:#e3f2fd
```

### When to Use Batch GD:

```mermaid
graph LR
    A[Use Batch GD When] --> B[Small Dataset < 10K]
    A --> C[Need Accurate Gradients]
    A --> D[Stable Convergence Required]
    A --> E[GPU Memory Available]
```

## 2. Mathematical Formulation

### General Form for Linear Regression:

For n features and m samples:
```
J(θ) = (1/2m) Σᵢ₌₁ᵐ (hᵢ(θ) - yᵢ)²
```

Where:
```
hᵢ(θ) = θ₀ + θ₁x₁ᵢ + θ₂x₂ᵢ + ... + θₙxₙᵢ
```

### Gradient for Each Parameter:

```
∂J/∂θ₀ = (1/m) Σᵢ₌₁ᵐ (hᵢ(θ) - yᵢ)
∂J/∂θⱼ = (1/m) Σᵢ₌₁ᵐ (hᵢ(θ) - yᵢ) · xⱼᵢ
```

### Vectorized Form:

```
θ = θ - α · (1/m) · Xᵀ(Xθ - y)
```

### Derivation Steps:

1. Start with cost function
2. Take partial derivatives
3. Vectorize using matrix operations
4. Implement update rule

## 3. Matrix Calculus Intuition

```python
# Matrix form of linear regression
# X: (m, n) features
# θ: (n, 1) parameters
# y: (m, 1) targets

# Predictions (vectorized)
h = X @ θ

# Cost (vectorized)
J = (1/(2*m)) * (h - y).T @ (h - y)

# Gradient (vectorized)
∇J = (1/m) * X.T @ (h - y)
```

## 4. Algorithm Steps

```mermaid
graph TD
    A[Algorithm: Batch GD] --> B[Step 1: Initialize θ]
    B --> C[Step 2: For each epoch]
    C --> D[Step 3: Compute all predictions]
    D --> E[Step 4: Compute all errors]
    E --> F[Step 5: Compute gradients]
    F --> G[Step 6: Update parameters]
    G --> H[Step 7: Check convergence]
    H --> I[Step 8: Return θ]
```

## 5. Time Complexity

- **Per epoch**: O(m × n)
- **Total**: O(epochs × m × n)
- **Space**: O(m × n) for storing data

## 6. Space Complexity

- **Data storage**: O(m × n)
- **Gradient storage**: O(n)
- **Total**: O(m × n)

## 7. Building GDRegressor Class

```python
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import time

class BatchGDRegressor:
    """
    Batch Gradient Descent Regressor
    
    Parameters:
    -----------
    learning_rate : float, default=0.01
        Step size for parameter updates
    epochs : int, default=100
        Maximum number of iterations
    tolerance : float, default=1e-4
        Convergence threshold
    add_intercept : bool, default=True
        Whether to add intercept term
    """
    
    def __init__(self, learning_rate=0.01, epochs=100, tolerance=1e-4, add_intercept=True):
        self.lr = learning_rate
        self.epochs = epochs
        self.tolerance = tolerance
        self.add_intercept = add_intercept
        self.coef_ = None
        self.intercept_ = None
        self.loss_history = []
        self.gradient_history = []
    
    def _add_intercept_term(self, X):
        """Add a column of ones for intercept"""
        if self.add_intercept:
            return np.c_[np.ones((X.shape[0], 1)), X]
        return X
    
    def fit(self, X_train, y_train):
        """
        Train the model using batch gradient descent
        
        Parameters:
        -----------
        X_train : array-like, shape (n_samples, n_features)
            Training features
        y_train : array-like, shape (n_samples,)
            Target values
        """
        # Add intercept term
        X = self._add_intercept_term(X_train)
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.theta_ = np.zeros(n_features)
        
        # Gradient descent loop
        for epoch in range(self.epochs):
            # Make predictions
            y_pred = np.dot(X, self.theta_)
            
            # Compute loss
            error = y_train - y_pred
            loss = np.mean(error**2)
            self.loss_history.append(loss)
            
            # Compute gradient
            gradient = -(2/n_samples) * np.dot(X.T, error)
            self.gradient_history.append(np.linalg.norm(gradient))
            
            # Check convergence
            if epoch > 0 and abs(self.loss_history[-2] - loss) < self.tolerance:
                print(f"Converged at epoch {epoch}")
                break
            
            # Update parameters
            self.theta_ -= self.lr * gradient
            
            # Print progress
            if epoch % 50 == 0:
                print(f"Epoch {epoch}: Loss = {loss:.6f}")
        
        # Extract intercept and coefficients
        if self.add_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0
            self.coef_ = self.theta_
        
        return self
    
    def predict(self, X_test):
        """
        Make predictions
        
        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test features
            
        Returns:
        --------
        y_pred : array-like, shape (n_samples,)
            Predicted values
        """
        if self.add_intercept:
            X = np.c_[np.ones((X_test.shape[0], 1)), X_test]
        else:
            X = X_test
        return np.dot(X, self.theta_)
    
    def score(self, X_test, y_test):
        """
        Calculate R² score
        
        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test features
        y_test : array-like, shape (n_samples,)
            True values
            
        Returns:
        --------
        r2 : float
            R² score
        """
        y_pred = self.predict(X_test)
        return r2_score(y_test, y_pred)
    
    def plot_loss(self, figsize=(10, 6)):
        """Plot loss history"""
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        
        # Plot loss
        axes[0].plot(self.loss_history)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss (MSE)')
        axes[0].set_title('Training Loss History')
        axes[0].grid(True)
        axes[0].set_yscale('log')
        
        # Plot gradient norm
        axes[1].plot(self.gradient_history)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Gradient Norm')
        axes[1].set_title('Gradient Norm History')
        axes[1].grid(True)
        axes[1].set_yscale('log')
        
        plt.tight_layout()
        plt.show()
        return fig, axes
    
    def get_params(self):
        """Return model parameters"""
        return {
            'coef_': self.coef_,
            'intercept_': self.intercept_
        }
```

## 8. Visualization: Training Animation

```python
def animate_batch_gd(X, y, model):
    """Animate batch gradient descent convergence"""
    
    X_with_intercept = np.c_[np.ones((X.shape[0], 1)), X]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: Regression line
    ax1.scatter(X, y, alpha=0.5, label='Data')
    x_line = np.linspace(X.min() - 1, X.max() + 1, 100)
    line, = ax1.plot([], [], 'r-', linewidth=2)
    ax1.set_xlabel('X')
    ax1.set_ylabel('y')
    ax1.set_title('Batch GD: Regression Line Evolution')
    ax1.legend()
    ax1.grid(True)
    
    # Right: Loss history
    loss_line, = ax2.plot([], [], 'b-', linewidth=2)
    ax2.set_xlim(0, len(model.loss_history))
    ax2.set_ylim(0, max(model.loss_history) * 1.1)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Loss Convergence')
    ax2.grid(True)
    
    def update(frame):
        # Get parameters at this frame
        theta = model.theta_history[frame]
        m = theta[1] if model.add_intercept else theta[0]
        b = theta[0] if model.add_intercept else 0
        
        # Update regression line
        y_line = m * x_line + b
        line.set_data(x_line, y_line)
        
        # Update loss plot
        x_data = list(range(frame + 1))
        y_data = model.loss_history[:frame + 1]
        loss_line.set_data(x_data, y_data)
        
        ax1.set_title(f'Epoch {frame+1}: m={m:.3f}, b={b:.3f}')
        return line, loss_line
    
    anim = animation.FuncAnimation(
        fig, update,
        frames=len(model.loss_history),
        interval=100,
        blit=True
    )
    
    plt.close()
    return anim
```

## 9. Comparison with Sklearn

```python
from sklearn.linear_model import LinearRegression

def compare_with_sklearn(X, y):
    """Compare our implementation with sklearn"""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Our implementation
    start_time = time.time()
    bgd = BatchGDRegressor(learning_rate=0.01, epochs=500)
    bgd.fit(X_train, y_train)
    our_time = time.time() - start_time
    our_score = bgd.score(X_test, y_test)
    
    # Sklearn implementation
    start_time = time.time()
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    sklearn_time = time.time() - start_time
    sklearn_score = lr.score(X_test, y_test)
    
    # Compare results
    print("=" * 50)
    print("Comparison: Our BGD vs Sklearn")
    print("=" * 50)
    print(f"Our BGD - R²: {our_score:.4f}, Time: {our_time:.4f}s")
    print(f"Sklearn - R²: {sklearn_score:.4f}, Time: {sklearn_time:.4f}s")
    print("\nParameters:")
    print(f"Our BGD: coef={bgd.coef_}, intercept={bgd.intercept_}")
    print(f"Sklearn: coef={lr.coef_}, intercept={lr.intercept_}")
    
    return bgd, lr

# Load data and compare
X, y = load_diabetes(return_X_y=True)
bgd, lr = compare_with_sklearn(X, y)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss history
axes[0].plot(bgd.loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Batch GD Loss History')
axes[0].set_yscale('log')
axes[0].grid(True)

# Plot 2: Coefficient comparison
features = range(len(bgd.coef_))
axes[1].bar(features, bgd.coef_, alpha=0.6, label='Our BGD')
axes[1].bar(features, lr.coef_, alpha=0.6, label='Sklearn')
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Coefficient Value')
axes[1].set_title('Coefficient Comparison')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
```

## 10. Pros & Cons of Batch GD

### Advantages:
1. **Accurate**: Uses exact gradient
2. **Stable**: Smooth convergence
3. **Deterministic**: Same results each run
4. **Efficient**: For small datasets

### Disadvantages:
1. **Slow**: For large datasets
2. **Memory**: Requires storing all data
3. **Local Minima**: Can get stuck

## 11. Interview Questions

### Q1: What happens when batch size equals dataset size?
Answer: This is Batch Gradient Descent.

### Q2: How does batch size affect convergence?
Larger batch → smoother convergence, slower updates
Smaller batch → noisier convergence, faster updates

### Q3: Why use Batch GD over SGD?
When dataset is small and accuracy is important

## 12. Exercises

### Exercise 1: Implement Early Stopping
```python
def early_stopping_bgd(X_train, y_train, X_val, y_val, patience=10):
    """Implement Batch GD with early stopping"""
    pass  # Your implementation
```

### Exercise 2: Add L2 Regularization
```python
def ridge_bgd(X, y, alpha=0.1):
    """Implement Batch GD with L2 regularization"""
    pass  # Your implementation
```
```

---

## Notebook 3: Stochastic Gradient Descent

```markdown
# Stochastic Gradient Descent: Complete Implementation

## 1. Why Batch GD Becomes Slow

Batch GD becomes impractical when:
- Dataset has millions of samples
- Memory is limited
- Need real-time updates

```mermaid
graph TD
    A[Batch GD Problems] --> B[O(m×n) per iteration]
    A --> C[Memory: O(m×n)]
    A --> D[Slow for Large Data]
    A --> E[Stuck in Local Minima]
    
    B --> F[10M samples × 100 features]
    C --> G[Memory Overflow]
    D --> H[Days/weeks to train]
    E --> I[Deterministic Updates]
```

## 2. SGD Intuition

Stochastic Gradient Descent updates parameters using **one random sample** at each step.

### Random Sampling:

```mermaid
graph LR
    A[Dataset] --> B[Random Sample]
    B --> C[Compute Gradient]
    C --> D[Update Parameters]
    D --> E[Repeat]
    
    style B fill:#ffeb3b
    style C fill:#4caf50
```

### Noisy Updates:

SGD is like a drunk person finding their way home:
- Takes erratic steps
- Might wander but eventually gets there
- Can explore more of the landscape

## 3. Mathematical Formulation

### Update Rule:
```
θ = θ - α · ∇J(θ; x^(i), y^(i))
```

Where (x^(i), y^(i)) is a randomly selected training example.

### Gradient for Single Sample:
```
∇J(θ) = (h(x^(i)) - y^(i)) · x^(i)
```

### Comparison:
```mermaid
graph LR
    A[Batch GD] --> B[θ = θ - α·(1/m)Σ∇Jᵢ]
    C[SGD] --> D[θ = θ - α·∇Jᵢ]
    
    B --> E[Accurate Direction]
    D --> F[Approximate Direction]
```

## 4. Learning Schedules

### Types of Learning Rate Schedules:

```mermaid
graph TD
    A[Learning Schedules] --> B[Constant]
    A --> C[Time Decay]
    A --> D[Step Decay]
    A --> E[Exponential]
    A --> F[Inverse Scaling]
    
    B --> G[η = η₀]
    C --> H[η = η₀/(1+kt)]
    D --> I[η = η₀×0.5^(t/step)]
    E --> J[η = η₀×e^(-kt)]
    F --> K[η = η₀/(1+kt²)]
```

### Implementation:

```python
def learning_rate_schedule(initial_lr, epoch, schedule='time_decay'):
    """
    Different learning rate schedules
    
    Parameters:
    -----------
    initial_lr : float
        Initial learning rate
    epoch : int
        Current epoch
    schedule : str
        Type of schedule
        
    Returns:
    --------
    lr : float
        Current learning rate
    """
    
    if schedule == 'constant':
        return initial_lr
        
    elif schedule == 'time_decay':
        # η = η₀ / (1 + k·t)
        k = 0.01
        return initial_lr / (1 + k * epoch)
        
    elif schedule == 'step_decay':
        # Drop by half every 10 epochs
        drop_rate = 0.5
        epochs_drop = 10
        return initial_lr * (drop_rate ** (epoch // epochs_drop))
        
    elif schedule == 'exponential':
        # η = η₀ · e^(-k·t)
        k = 0.01
        return initial_lr * np.exp(-k * epoch)
        
    elif schedule == 'inverse_scaling':
        # η = η₀ / (1 + k·t²)
        k = 0.001
        return initial_lr / (1 + k * epoch**2)
    
    else:
        raise ValueError(f"Unknown schedule: {schedule}")
```

## 5. Epochs and Iterations

```mermaid
graph TD
    A[SGD Terminology] --> B[Epoch]
    A --> C[Iteration]
    A --> D[Batch]
    
    B --> E[One pass through all data]
    C --> F[One update step]
    D --> G[Set of samples]
    
    E --> H[epochs × m iterations]
```

## 6. Online Learning

SGD enables **online learning**:
- Update model with new data as it arrives
- No need to retrain from scratch
- Great for streaming data

```mermaid
graph LR
    A[New Data Arrives] --> B[Update Model]
    B --> C[Continue Learning]
    C --> D[Adapt to Changes]
```

## 7. SGD from Scratch

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import time
import random

class SGDRegressor:
    """
    Stochastic Gradient Descent Regressor
    
    Parameters:
    -----------
    learning_rate : float, default=0.01
        Initial learning rate
    epochs : int, default=100
        Number of epochs
    schedule : str, default='time_decay'
        Learning rate schedule
    random_state : int, default=42
        Random seed
    """
    
    def __init__(self, learning_rate=0.01, epochs=100,
                 schedule='time_decay', random_state=42):
        self.initial_lr = learning_rate
        self.epochs = epochs
        self.schedule = schedule
        self.random_state = random_state
        np.random.seed(random_state)
        
        self.coef_ = None
        self.intercept_ = None
        self.loss_history = []
        self.lr_history = []
        self.gradient_norm_history = []
    
    def _get_lr(self, epoch):
        """Get learning rate for current epoch"""
        return learning_rate_schedule(self.initial_lr, epoch, self.schedule)
    
    def fit(self, X_train, y_train):
        """
        Train model using SGD
        
        Parameters:
        -----------
        X_train : array-like, shape (n_samples, n_features)
            Training features
        y_train : array-like, shape (n_samples,)
            Target values
        """
        n_samples, n_features = X_train.shape
        
        # Initialize parameters
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0
        
        # Training loop
        for epoch in range(self.epochs):
            current_lr = self._get_lr(epoch)
            self.lr_history.append(current_lr)
            
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]
            
            epoch_loss = 0
            
            # Process each sample
            for i in range(n_samples):
                xi = X_shuffled[i]
                yi = y_shuffled[i]
                
                # Predict
                y_pred = np.dot(xi, self.coef_) + self.intercept_
                
                # Error
                error = yi - y_pred
                epoch_loss += error**2
                
                # Update parameters (stochastic gradient)
                self.coef_ += 2 * current_lr * error * xi
                self.intercept_ += 2 * current_lr * error
            
            # Store average loss
            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)
            
            # Store gradient norm
            self.gradient_norm_history.append(
                np.linalg.norm(self.coef_) + abs(self.intercept_)
            )
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: LR={current_lr:.5f}, Loss={avg_loss:.4f}")
        
        return self
    
    def predict(self, X_test):
        """Make predictions"""
        return np.dot(X_test, self.coef_) + self.intercept_
    
    def score(self, X_test, y_test):
        """Calculate R² score"""
        y_pred = self.predict(X_test)
        return r2_score(y_test, y_pred)
    
    def plot_training(self, figsize=(15, 10)):
        """Plot training metrics"""
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        
        # Plot 1: Loss history
        axes[0, 0].plot(self.loss_history)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss (MSE)')
        axes[0, 0].set_title('Loss History')
        axes[0, 0].grid(True)
        axes[0, 0].set_yscale('log')
        
        # Plot 2: Learning rate
        axes[0, 1].plot(self.lr_history)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Learning Rate')
        axes[0, 1].set_title('Learning Rate Schedule')
        axes[0, 1].grid(True)
        
        # Plot 3: Gradient norm
        axes[1, 0].plot(self.gradient_norm_history)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Gradient Norm')
        axes[1, 0].set_title('Gradient Norm History')
        axes[1, 0].grid(True)
        
        # Plot 4: Coefficients
        axes[1, 1].bar(range(len(self.coef_)), self.coef_)
        axes[1, 1].set_xlabel('Feature Index')
        axes[1, 1].set_ylabel('Coefficient Value')
        axes[1, 1].set_title('Final Coefficients')
        axes[1, 1].grid(True)
        
        plt.tight_layout()
        plt.show()
        return fig, axes
```

## 8. Visualization: SGD Animation

```python
from matplotlib import animation
from IPython.display import HTML

def animate_sgd_convergence(X, y, model):
    """Animate SGD convergence"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: Data and regression line
    ax1.scatter(X, y, alpha=0.5, label='Data')
    x_line = np.linspace(X.min() - 1, X.max() + 1, 100)
    line, = ax1.plot([], [], 'r-', linewidth=2)
    ax1.set_xlabel('X')
    ax1.set_ylabel('y')
    ax1.set_title('SGD: Regression Line Evolution')
    ax1.legend()
    ax1.grid(True)
    
    # Right: Loss history
    loss_line, = ax2.plot([], [], 'b-', linewidth=2)
    ax2.set_xlim(0, len(model.loss_history))
    ax2.set_ylim(0, max(model.loss_history) * 1.1)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Loss Convergence (Noisy)')
    ax2.grid(True)
    
    def update(frame):
        # Get current parameters
        # Note: We're using the final model here
        # In practice, store parameter history
        m = model.coef_[0] if len(model.coef_) == 1 else 0
        b = model.intercept_
        
        # Update regression line
        y_line = m * x_line + b
        line.set_data(x_line, y_line)
        
        # Update loss
        x_data = list(range(frame + 1))
        y_data = model.loss_history[:frame + 1]
        loss_line.set_data(x_data, y_data)
        
        ax1.set_title(f'Epoch {frame+1}: LR={model.lr_history[frame]:.4f}')
        return line, loss_line
    
    anim = animation.FuncAnimation(
        fig, update,
        frames=len(model.loss_history),
        interval=100,
        blit=True
    )
    
    plt.close()
    return anim

# Test with simple data
X_simple = np.linspace(-5, 5, 100).reshape(-1, 1)
y_simple = 2 * X_simple.ravel() + 3 + np.random.randn(100) * 2

model = SGDRegressor(learning_rate=0.1, epochs=100, schedule='time_decay')
model.fit(X_simple, y_simple)

anim = animate_sgd_convergence(X_simple, y_simple, model)
HTML(anim.to_html5_video())
```

## 9. Time Comparison

```python
def time_comparison(X, y):
    """Compare training time between Batch GD and SGD"""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    results = {}
    
    # Batch GD
    print("Training Batch GD...")
    start = time.time()
    bgd = BatchGDRegressor(learning_rate=0.01, epochs=100)
    bgd.fit(X_train, y_train)
    bgd_time = time.time() - start
    bgd_score = bgd.score(X_test, y_test)
    
    # SGD
    print("Training SGD...")
    start = time.time()
    sgd = SGDRegressor(learning_rate=0.01, epochs=100, schedule='constant')
    sgd.fit(X_train, y_train)
    sgd_time = time.time() - start
    sgd_score = sgd.score(X_test, y_test)
    
    # Results
    results = {
        'Batch GD': {'time': bgd_time, 'score': bgd_score},
        'SGD': {'time': sgd_time, 'score': sgd_score}
    }
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Time comparison
    methods = list(results.keys())
    times = [results[m]['time'] for m in methods]
    axes[0].bar(methods, times, color=['blue', 'orange'])
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title('Training Time Comparison')
    axes[0].grid(True)
    
    # R² comparison
    scores = [results[m]['score'] for m in methods]
    axes[1].bar(methods, scores, color=['blue', 'orange'])
    axes[1].set_ylabel('R² Score')
    axes[1].set_title('Model Performance Comparison')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Run comparison
results = time_comparison(X, y)
print("\nResults:")
for method, data in results.items():
    print(f"{method}: Time={data['time']:.3f}s, R²={data['score']:.4f}")
```

## 10. Learning Rate Schedules Comparison

```python
def compare_schedules(X, y):
    """Compare different learning rate schedules"""
    
    schedules = ['constant', 'time_decay', 'step_decay', 'exponential']
    results = {}
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for i, schedule in enumerate(schedules):
        ax = axes[i//2, i%2]
        
        model = SGDRegressor(
            learning_rate=0.1,
            epochs=100,
            schedule=schedule
        )
        model.fit(X, y)
        
        # Plot loss
        ax.plot(model.loss_history, label='Loss')
        ax.plot(model.lr_history, label='Learning Rate', alpha=0.7)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Value')
        ax.set_title(f'Schedule: {schedule}')
        ax.legend()
        ax.grid(True)
        ax.set_yscale('log')
        
        results[schedule] = {
            'final_loss': model.loss_history[-1],
            'final_lr': model.lr_history[-1]
        }
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare schedules
schedule_results = compare_schedules(X, y)
print("\nSchedule Comparison:")
for schedule, data in schedule_results.items():
    print(f"{schedule}: Final Loss={data['final_loss']:.4f}, Final LR={data['final_lr']:.4f}")
```

## 11. When to Use SGD

```mermaid
graph TD
    A[Use SGD When] --> B[Large Dataset > 1M]
    A --> C[Need Fast Training]
    A --> D[Online Learning]
    A --> E[Limited Memory]
    A --> F[Escaping Local Minima]
    
    B --> G[Memory Efficient]
    C --> H[Real-time Updates]
    D --> I[Streaming Data]
    E --> J[One Sample at a Time]
    F --> K[Randomness Helps]
```

## 12. Practical Tips

### 1. Feature Scaling is Critical
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

### 2. Use Learning Rate Schedules
Start with higher LR and decrease over time

### 3. Shuffle Data Each Epoch
Prevents learning from order

### 4. Monitor Training Loss
Should decrease roughly

## 13. Interview Questions

### Q1: Why does SGD use random samples?
To avoid cycles and escape local minima

### Q2: What are the trade-offs of SGD?
Speed vs. Accuracy trade-off

### Q3: How does learning rate decay help?
Helps stabilize convergence

## 14. Advantages and Limitations

### Advantages:
- Fast for large datasets
- Memory efficient
- Online learning capable
- Escapes local minima

### Limitations:
- Noisy updates
- May never fully converge
- Sensitive to learning rate

## 15. Exercise

### Exercise: Implement SGD with Momentum
```python
def sgd_with_momentum(X, y, lr=0.01, momentum=0.9, epochs=100):
    """Implement SGD with momentum"""
    pass  # Your implementation
```
```

---

## Notebook 4: Mini-Batch Gradient Descent

```markdown
# Mini-Batch Gradient Descent: Complete Implementation

## 1. Why Mini-Batch?

Mini-Batch GD combines the best of Batch GD and SGD:

```mermaid
graph LR
    A[Batch Size Comparison] --> B[Batch GD: All Data]
    A --> C[SGD: 1 Sample]
    A --> D[Mini-Batch: 32-512 Samples]
    
    B --> E[Accurate, Slow]
    C --> F[Fast, Noisy]
    D --> G[Balanced]
    
    style D fill:#c8e6c9
```

### The Goldilocks Choice:
- Not too big (like Batch)
- Not too small (like SGD)
- **Just right** for most applications

## 2. What is Mini-Batch?

```mermaid
graph TD
    A[Mini-Batch Process] --> B[Shuffle Data]
    B --> C[Split into Batches]
    C --> D[For Each Batch]
    D --> E[Forward Pass]
    E --> F[Compute Gradient]
    F --> G[Update Parameters]
    G --> H[Next Batch]
```

### Key Characteristics:
- Uses **subset** of data
- Typical batch sizes: 32, 64, 128, 256
- Vectorized operations
- Balanced convergence

## 3. Batch Size Selection

### Why 32, 64, 128, 256?

```mermaid
graph TD
    A[Batch Size Factors] --> B[Memory]
    A --> C[GPU Architecture]
    A --> D[Convergence]
    
    B --> E[RAM/VRAM Limits]
    C --> F[Power of 2]
    D --> G[Stable Updates]
    
    F --> H[GPU Optimized]
    H --> I[32, 64, 128, 256]
```

### GPU Optimization:

GPUs are designed for matrix operations:
- 32 = Warp size
- 64 = 2 warps
- 128 = 4 warps
- 256 = 8 warps

## 4. Mathematical Derivation

### Update Rule:
```
θ = θ - α · (1/b) · Σᵢ₌₁ᵇ ∇J(θ; xᵢ, yᵢ)
```

Where b is the batch size.

### Comparison:

| Method | Batch Size | Update Formula |
|--------|-----------|----------------|
| Batch GD | m | θ = θ - α·(1/m)·Σᵢ₌₁ᵐ ∇Jᵢ |
| SGD | 1 | θ = θ - α·∇Jᵢ |
| Mini-Batch | b | θ = θ - α·(1/b)·Σᵢ₌₁ᵇ ∇Jᵢ |

## 5. Algorithm Steps

```python
def mini_batch_gd_algorithm(X, y, batch_size=32, learning_rate=0.01, epochs=100):
    """
    Mini-Batch Gradient Descent Algorithm
    
    Steps:
    1. Initialize parameters
    2. For each epoch:
        a. Shuffle dataset
        b. Split into batches
        c. For each batch:
            - Compute gradient
            - Update parameters
    3. Return optimized parameters
    """
    pass
```

## 6. Implementation

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import time
import random

class MiniBatchGDRegressor:
    """
    Mini-Batch Gradient Descent Regressor
    
    Parameters:
    -----------
    batch_size : int, default=32
        Size of each mini-batch
    learning_rate : float, default=0.01
        Learning rate
    epochs : int, default=100
        Number of epochs
    schedule : str, default='constant'
        Learning rate schedule
    random_state : int, default=42
        Random seed
    """
    
    def __init__(self, batch_size=32, learning_rate=0.01, epochs=100,
                 schedule='constant', random_state=42):
        self.batch_size = batch_size
        self.initial_lr = learning_rate
        self.epochs = epochs
        self.schedule = schedule
        self.random_state = random_state
        np.random.seed(random_state)
        
        self.coef_ = None
        self.intercept_ = None
        self.loss_history = []
        self.batch_loss_history = []
        self.lr_history = []
    
    def _get_lr(self, epoch):
        """Get learning rate for current epoch"""
        if self.schedule == 'constant':
            return self.initial_lr
        elif self.schedule == 'time_decay':
            return self.initial_lr / (1 + 0.01 * epoch)
        elif self.schedule == 'step_decay':
            return self.initial_lr * (0.5 ** (epoch // 10))
        else:
            return self.initial_lr
    
    def _create_batches(self, X, y):
        """Create mini-batches"""
        n_samples = X.shape[0]
        indices = np.random.permutation(n_samples)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        batches = []
        for i in range(0, n_samples, self.batch_size):
            end = min(i + self.batch_size, n_samples)
            X_batch = X_shuffled[i:end]
            y_batch = y_shuffled[i:end]
            batches.append((X_batch, y_batch))
        
        return batches
    
    def fit(self, X_train, y_train):
        """
        Train model using mini-batch gradient descent
        
        Parameters:
        -----------
        X_train : array-like, shape (n_samples, n_features)
            Training features
        y_train : array-like, shape (n_samples,)
            Target values
        """
        n_samples, n_features = X_train.shape
        
        # Initialize parameters
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0
        
        # Training loop
        for epoch in range(self.epochs):
            current_lr = self._get_lr(epoch)
            self.lr_history.append(current_lr)
            
            # Create mini-batches
            batches = self._create_batches(X_train, y_train)
            
            epoch_loss = 0
            num_batches = len(batches)
            
            # Process each batch
            for X_batch, y_batch in batches:
                batch_size = X_batch.shape[0]
                
                # Predict
                y_pred = np.dot(X_batch, self.coef_) + self.intercept_
                
                # Compute gradients
                error = y_batch - y_pred
                batch_loss = np.mean(error**2)
                self.batch_loss_history.append(batch_loss)
                epoch_loss += batch_loss
                
                # Update parameters (batch gradient)
                coef_grad = -(2/batch_size) * np.dot(X_batch.T, error)
                intercept_grad = -(2/batch_size) * np.sum(error)
                
                self.coef_ -= current_lr * coef_grad
                self.intercept_ -= current_lr * intercept_grad
            
            # Store average epoch loss
            avg_loss = epoch_loss / num_batches
            self.loss_history.append(avg_loss)
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: LR={current_lr:.5f}, Loss={avg_loss:.4f}")
        
        return self
    
    def predict(self, X_test):
        """Make predictions"""
        return np.dot(X_test, self.coef_) + self.intercept_
    
    def score(self, X_test, y_test):
        """Calculate R² score"""
        y_pred = self.predict(X_test)
        return r2_score(y_test, y_pred)
    
    def plot_training(self, figsize=(15, 10)):
        """Plot training metrics"""
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        
        # Plot 1: Epoch loss
        axes[0, 0].plot(self.loss_history, 'b-', linewidth=2, label='Epoch Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Epoch Loss History')
        axes[0, 0].grid(True)
        axes[0, 0].set_yscale('log')
        axes[0, 0].legend()
        
        # Plot 2: Batch loss
        axes[0, 1].plot(self.batch_loss_history, 'r-', alpha=0.5, linewidth=0.5)
        axes[0, 1].set_xlabel('Batch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Batch Loss History (Noisy)')
        axes[0, 1].grid(True)
        axes[0, 1].set_yscale('log')
        
        # Plot 3: Learning rate
        axes[1, 0].plot(self.lr_history)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_title('Learning Rate Schedule')
        axes[1, 0].grid(True)
        
        # Plot 4: Final coefficients
        axes[1, 1].bar(range(len(self.coef_)), self.coef_)
        axes[1, 1].set_xlabel('Feature Index')
        axes[1, 1].set_ylabel('Coefficient Value')
        axes[1, 1].set_title('Final Coefficients')
        axes[1, 1].grid(True)
        axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        return fig, axes
```

## 7. Animation: Mini-Batch vs Others

```python
from matplotlib import animation
from IPython.display import HTML

def compare_convergence_animation(X, y):
    """Animate convergence comparison between GD variants"""
    
    # Initialize models
    bgd_model = BatchGDRegressor(learning_rate=0.01, epochs=100)
    sgd_model = SGDRegressor(learning_rate=0.01, epochs=100, schedule='constant')
    mbgd_model = MiniBatchGDRegressor(batch_size=32, learning_rate=0.01, epochs=100)
    
    # Train models
    bgd_model.fit(X, y)
    sgd_model.fit(X, y)
    mbgd_model.fit(X, y)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Setup each subplot
    colors = ['blue', 'red', 'green']
    titles = ['Batch GD', 'SGD', 'Mini-Batch GD']
    histories = [bgd_model.loss_history, sgd_model.loss_history, mbgd_model.loss_history]
    
    for ax, color, title, hist in zip(axes, colors, titles, histories):
        line, = ax.plot([], [], color=color, linewidth=2)
        ax.set_xlim(0, len(hist))
        ax.set_ylim(0, max(hist) * 1.1)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(title)
        ax.grid(True)
        ax.set_yscale('log')
    
    def update(frame):
        for i, (ax, hist) in enumerate(zip(axes, histories)):
            x_data = list(range(frame + 1))
            y_data = hist[:frame + 1]
            ax.lines[0].set_data(x_data, y_data)
        return axes
    
    anim = animation.FuncAnimation(
        fig, update,
        frames=max(len(h) for h in histories),
        interval=100,
        blit=True
    )
    
    plt.close()
    return anim
```

## 8. Performance Benchmarking

```python
def benchmark_batch_sizes(X, y):
    """Compare performance across different batch sizes"""
    
    batch_sizes = [1, 8, 16, 32, 64, 128, 256, 512]
    results = {
        'time': [],
        'score': [],
        'final_loss': []
    }
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    for batch_size in batch_sizes:
        print(f"\nTraining with batch size: {batch_size}")
        
        # Train model
        start = time.time()
        model = MiniBatchGDRegressor(
            batch_size=batch_size,
            learning_rate=0.01,
            epochs=100
        )
        model.fit(X_train, y_train)
        train_time = time.time() - start
        
        # Evaluate
        score = model.score(X_test, y_test)
        final_loss = model.loss_history[-1]
        
        results['time'].append(train_time)
        results['score'].append(score)
        results['final_loss'].append(final_loss)
        
        print(f"Time: {train_time:.3f}s, R²: {score:.4f}")
    
    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].plot(batch_sizes, results['time'], 'bo-')
    axes[0].set_xlabel('Batch Size')
    axes[0].set_ylabel('Training Time (s)')
    axes[0].set_title('Time vs Batch Size')
    axes[0].grid(True)
    axes[0].set_xscale('log')
    
    axes[1].plot(batch_sizes, results['score'], 'go-')
    axes[1].set_xlabel('Batch Size')
    axes[1].set_ylabel('R² Score')
    axes[1].set_title('Accuracy vs Batch Size')
    axes[1].grid(True)
    axes[1].set_xscale('log')
    
    axes[2].plot(batch_sizes, results['final_loss'], 'ro-')
    axes[2].set_xlabel('Batch Size')
    axes[2].set_ylabel('Final Loss')
    axes[2].set_title('Loss vs Batch Size')
    axes[2].grid(True)
    axes[2].set_xscale('log')
    axes[2].set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Run benchmark
benchmark_results = benchmark_batch_sizes(X, y)
```

## 9. Comparison with Batch and SGD

```python
def comprehensive_comparison(X, y):
    """Comprehensive comparison of all GD variants"""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    variants = {
        'Batch GD': BatchGDRegressor(learning_rate=0.01, epochs=100),
        'SGD': SGDRegressor(learning_rate=0.01, epochs=100, schedule='constant'),
        'Mini-Batch (32)': MiniBatchGDRegressor(batch_size=32, learning_rate=0.01, epochs=100),
        'Mini-Batch (128)': MiniBatchGDRegressor(batch_size=128, learning_rate=0.01, epochs=100),
        'Mini-Batch (512)': MiniBatchGDRegressor(batch_size=512, learning_rate=0.01, epochs=100)
    }
    
    results = {}
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()
    
    for i, (name, model) in enumerate(variants.items()):
        ax = axes[i]
        
        # Train model
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        
        # Evaluate
        score = model.score(X_test, y_test)
        
        # Store results
        results[name] = {
            'time': train_time,
            'score': score,
            'loss': model.loss_history
        }
        
        # Plot loss history
        ax.plot(model.loss_history)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'{name}\nTime: {train_time:.3f}s, R²: {score:.4f}')
        ax.grid(True)
        ax.set_yscale('log')
    
    # Hide extra subplot
    axes[-1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "=" * 60)
    print("COMPREHENSIVE COMPARISON: Gradient Descent Variants")
    print("=" * 60)
    for name, data in results.items():
        print(f"\n{name}:")
        print(f"  Training Time: {data['time']:.3f} seconds")
        print(f"  R² Score: {data['score']:.4f}")
        print(f"  Final Loss: {data['loss'][-1]:.4f}")
    
    return results

# Run comprehensive comparison
comparison_results = comprehensive_comparison(X, y)
```

## 10. Choosing Batch Size

```mermaid
graph TD
    A[Batch Size Selection] --> B[Small Batch]
    A --> C[Medium Batch]
    A --> D[Large Batch]
    
    B --> E[< 32]
    B --> F[More Noise]
    B --> G[Better Generalization]
    
    C --> H[32 - 256]
    C --> I[Balanced]
    C --> J[Recommended]
    
    D --> K[> 512]
    D --> L[Less Noise]
    D --> M[Less Generalization]
```

### Practical Recommendations:

1. **Start with 32 or 64**
2. **Increase if memory allows**
3. **Monitor convergence**
4. **Use power of 2** (GPU friendly)

## 11. Implementation Tips

```python
# 1. Use Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Use Learning Rate Decay
def adaptive_lr(epoch, initial_lr=0.01):
    return initial_lr / (1 + 0.01 * epoch)

# 3. Shuffle Each Epoch
indices = np.random.permutation(n_samples)
X_shuffled = X[indices]

# 4. Monitor Validation Loss
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)
```

## 12. Common Mistakes

### 1. Not Shuffling Data
❌ Using data in original order
✅ Shuffle each epoch

### 2. Wrong Batch Size
❌ Too large or too small
✅ Start with 32, tune if needed

### 3. No Learning Rate Schedule
❌ Constant learning rate
✅ Use decay schedule

### 4. Not Scaling Features
❌ Raw features
✅ Standardize or normalize

## 13. Performance Tips

1. **Vectorize operations**: Use numpy matrix operations
2. **Use appropriate data types**: Float32 for memory
3. **Parallelize**: Process batches in parallel
4. **GPU acceleration**: Use CuPy or PyTorch

## 14. Interview Questions

### Q1: How do you choose batch size?
Start with 32, tune based on:
- Memory constraints
- Dataset size
- Convergence speed

### Q2: What's the effect of batch size on training?
- Larger = smoother convergence, slower
- Smaller = noisier, faster

### Q3: Why are batch sizes powers of 2?
GPU architecture optimization

## 15. Advantages and Limitations

### Advantages:
- Balanced performance
- Vectorized operations
- Good generalization
- Memory efficient

### Limitations:
- Batch size tuning required
- Still can get stuck in local minima

## 16. Exercises

### Exercise 1: Automatic Batch Size Selection
```python
def find_optimal_batch_size(X, y):
    """Find optimal batch size automatically"""
    pass  # Your implementation
```

### Exercise 2: Adaptive Mini-Batch
```python
def adaptive_minibatch(X, y):
    """Implement adaptive batch size that changes during training"""
    pass  # Your implementation
```

## Summary Table

| Method | Batch Size | Speed | Stability | Memory | Best For |
|--------|-----------|-------|-----------|--------|----------|
| **Batch GD** | All samples | Slow | High | High | Small datasets |
| **SGD** | 1 | Fast | Low | Low | Large datasets |
| **Mini-Batch** | 32-512 | Medium | Medium | Medium | Most applications |

## Final Recommendations

```python
def recommend_gd_method(X_train):
    """Recommend GD method based on dataset size"""
    n_samples = X_train.shape[0]
    
    if n_samples < 1000:
        return "Batch GD"
    elif n_samples < 10000:
        return "Mini-Batch GD (batch_size=32-64)"
    elif n_samples < 100000:
        return "Mini-Batch GD (batch_size=128-256)"
    else:
        return "SGD with learning rate schedule"
```

## Complete Working Example

```python
# Complete mini-batch training pipeline
def complete_minibatch_pipeline(X, y):
    """Complete mini-batch GD pipeline"""
    
    # 1. Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # 2. Scale features
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 3. Train model
    mbgd = MiniBatchGDRegressor(
        batch_size=64,
        learning_rate=0.01,
        epochs=200,
        schedule='time_decay'
    )
    mbgd.fit(X_train_scaled, y_train)
    
    # 4. Evaluate
    train_score = mbgd.score(X_train_scaled, y_train)
    test_score = mbgd.score(X_test_scaled, y_test)
    
    print(f"Train R²: {train_score:.4f}")
    print(f"Test R²: {test_score:.4f}")
    
    # 5. Plot results
    mbgd.plot_training()
    
    return mbgd, scaler

# Run complete pipeline
model, scaler = complete_minibatch_pipeline(X, y)
```

This concludes the complete guide to Gradient Descent from scratch through Mini-Batch implementation. Each notebook builds on the previous ones, starting from basic concepts and gradually adding complexity and optimization techniques.
```